# Pipeline Step 03: Semantic Classification (LLMs)

This notebook imports our `asyncio` LLM batching logic from `src/classification.py`.
**Caching Guarantee**: It will check if `cascade_final_classifications.csv` exists and safely resume from where it left off, preventing duplicate API calls.

In [3]:
import os
import pandas as pd
import sys

# Ensure we can import from src/
sys.path.append(os.path.abspath(os.path.join("..", "..")))
from src.classification import (
    init_vertex_ai,
    get_model,
    pre_flight_cost_estimate,
    run_cascade_batch_async
)

BASE_DIR = os.path.abspath(os.path.join("..", "..")) + "/"
CANDIDATES_CSV = os.path.join(BASE_DIR, "data/processed/llm_candidate_queue.csv")
OUTPUT_CSV = os.path.join(BASE_DIR, "data/processed/cascade_final_classifications.csv")

print("Ready.")

Ready.


## 1. Load the Queue and Check Cache

In [4]:
df_queue = pd.read_csv(CANDIDATES_CSV)
df_queue["id"] = df_queue["id"].astype(str)

processed_ids = set()
if os.path.exists(OUTPUT_CSV):
    df_existing = pd.read_csv(OUTPUT_CSV, usecols=["id"])
    processed_ids = set(df_existing["id"].astype(str))
    print(f"🔄 Resuming cascade run: Found {len(processed_ids):,} already classified candidates.")
else:
    TARGET_CATEGORIES = [
        "anti_establishment_stance", "hedged_suspicion", "personal_experience",
        "source_citation", "appeal_to_authority", "procedural_skepticism", "maverick_authority"
    ]
    headers = ["id", "text"] + TARGET_CATEGORIES
    pd.DataFrame(columns=headers).to_csv(OUTPUT_CSV, index=False)
    print("🚀 Starting fresh cascade batch run...")

df_remaining = df_queue[~df_queue["id"].isin(processed_ids)].copy()
print(f"📊 Candidates remaining to process: {len(df_remaining):,}\n")

🚀 Starting fresh cascade batch run...
📊 Candidates remaining to process: 85,159



## 2. Pre-Flight Estimate

In [5]:
if not df_remaining.empty:
    pre_flight_cost_estimate(df_remaining, text_column="text", model_type="flash")
else:
    print("✅ No remaining items in queue! Cost is $0.00")

=== PRE-FLIGHT COST ESTIMATE ===
Dataset Size:   85,159 rows
Model Selected: Gemini 1.5 FLASH
-----------------------------------
Est. Input Tokens:  24,826,476 tokens
Est. Output Tokens: 6,386,925 tokens
-----------------------------------
Est. Input Cost:  $1.8620
Est. Output Cost: $1.9161
TOTAL EST. COST:  $3.7781


## 3. Run Async Batch (Uncomment to execute)

In [ ]:
import asyncio

# init_vertex_ai()
# custom_model = get_model()

# if not df_remaining.empty:
#     await run_cascade_batch_async(df_remaining, custom_model, OUTPUT_CSV, processed_ids, concurrent_requests=10)
#     print("✅ Final Cloud Batch complete! Candidate queue processed.")
# else:
#     print("✅ Queue already complete. Nothing to run.")